# ML-11 — Ship the Paper

## Predicting Content Decline with Client-Held-Out Validation

**Author:** Raunit Singh  
**Track:** Machine Learning  
**Data source:** FlyRank ML Internship dataset

### Abstract
This paper asks whether page-level search and content signals can help identify content that is already associated with decline. The analysis uses an anonymized teaching dataset of 30,000 content rows from 32 clients and evaluates a Random Forest model using a client-grouped holdout so that test clients are completely unseen during training. A naive row split produced ROC AUC 0.763 and Average Precision 0.776, while the honest client-grouped split produced ROC AUC 0.600 and Average Precision 0.587. The gap shows why client overlap can make validation look more optimistic than a new-client test. The final output is used as a ranked review queue rather than an automated content decision system.



## 1. Research Question and Problem Statement

The practical question is: **Can a machine-learning model help identify pages that deserve human review for decline risk, while still being evaluated honestly across clients?**

This is a decision-support problem. The goal is not to claim that the model has discovered Google's ranking algorithm or that a particular page characteristic causes decline. The model is used to prioritize pages for investigation.

The study also asks a methodological question: **How much does the reported model performance change when validation is done on completely unseen clients instead of randomly splitting individual rows?**


## 2. Data

The executed analysis used the anonymized FlyRank teaching dataset after the same filtering used in the earlier model work: pages with positive 90-day impressions and content age of at least 90 days were retained, duplicate content IDs were removed, and the decline label was derived from `trend_direction`.

**Executed dataset:** 30,000 rows across 32 clients. The observed decline base rate was 54.21%.

The analysis in this repository uses the available anonymized teaching slice. It does **not** claim that this notebook independently processed the full 79-million-row production warehouse. The public-safe dataset is intentionally anonymized.

The label is `is_declining_label = (trend_direction == 'down')`. Because the label comes from `trend_direction`, target-derived fields are not allowed to become model features.


## 3. Methodology

### Features
The model uses page-level search, traffic, engagement, content-age, freshness, and categorical content signals. Target-derived fields and identifiers are excluded.

Excluded fields: `trend_direction`, `trend_pct`, `is_declining_label`, `content_id`, and `client_id`.

### Model
The final model is a Random Forest classifier with 300 trees, `min_samples_leaf=5`, `class_weight='balanced_subsample'`, and `random_state=42`.

### Validation design
The primary evaluation keeps all rows belonging to a client together. Twenty percent of clients are held out for testing. This prevents the same client's pages from appearing in both training and test data.

For comparison, a naive random row split was also evaluated. That split had 31 clients in both training and testing data, so it is not the primary estimate for performance on unseen clients.

### Leakage checks
The executed leakage audit passed. The five target-derived or identifier fields listed above were excluded before model fitting.


## 4. Results

| Validation design | ROC AUC | Average Precision | Test rows | Test clients | Client overlap |
|---|---:|---:|---:|---:|---:|
| Naive random row split | 0.763 | 0.776 | 6,000 | 31 | 31 |
| Client-grouped split | **0.600** | **0.587** | 6,163 | 7 | **0** |

The random row split looks stronger, but it allows the same clients to appear on both sides of the evaluation. The client-grouped result is lower and is the more relevant estimate for a new-client setting because the seven test clients were not used for training.

The model therefore provides useful ranking signal, but the result should not be presented as near-perfect prediction. The main lesson is that validation design materially affects the measured performance.


## 5. Research Findings and Interpretation

### Content lifecycle
The source analysis reports that growing pages were younger on average than declining pages: approximately 185 days versus 228 days. Average word count was nearly the same, about 1,487 versus 1,481 words.

**Interpretation:** page age is associated with the observed growth/decline groups in that analysis. This does not establish that younger content causes growth.

### Freshness
The source analysis reports a 31–90 day freshness-window growth-to-decline ratio of 5.43:1 and a separate refreshed-versus-stale comparison for older content.

**Interpretation:** these are observed cohort comparisons. Selection effects and other differences between groups cannot be ruled out from these comparisons alone, so they should not be described as proof of a causal freshness effect.


## 6. Ranked Recommendations

The model output is turned into a review queue. Each page receives a decline-risk score, a reason code, a content archetype, and a recommended action.

Typical action categories are:
- **Stale high-visibility page:** review whether a refresh is justified, then revalidate.
- **Ranking opportunity:** review search intent and on-page relevance.
- **Depth improvement candidate:** review content depth and topic coverage.
- **Refresh candidate:** check whether the page actually needs a freshness update.
- **Needs data review:** resolve missing or inconsistent signals before acting.
- **General decline-risk review:** carry out a normal human content review.

These recommendations are prioritisation aids. They are not automatic instructions and do not establish the cause of decline.


## 7. Limitations and Honest Framing

1. The executed model evaluation is based on 30,000 anonymized rows, not an independent run over the full 79-million-row production warehouse.
2. The client-grouped test contains seven held-out clients, so the result should be interpreted with that population in mind.
3. Model scores indicate estimated class risk, not causation.
4. Research comparisons describe associations and cohort differences; they do not prove that freshness, age, or another single factor caused the outcome.
5. The dataset does not provide real business costs, so the action queue does not claim a measured financial ROI.
6. Important actions require human review, especially changes that affect publishing, deletion, redirects, canonical tags, or sensitive claims.


## 8. Reproducibility

The work is organized in the GitHub repository. ML-08, ML-09, and ML-10 notebooks provide the model, validation audit, and action-playbook trail that this paper builds on.

The primary validation setup is deterministic with `random_state=42`. The ML-10 workflow was executed successfully through GitHub Actions, and its final self-check passed.

Repository: https://github.com/singhraunit2807/flyrank-ml-internship-starter


## 9. Acknowledgements and Data Credit

This work was completed as part of the FlyRank ML Internship. The dataset and internship materials are credited to FlyRank.

Data source: https://flyrank.ai
